In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk

In [2]:
#Universal Constants
c=3*10**8
mu_0 = 4*(np.pi)*(10**-7)
epsilon = 8.854*10**-12
hbar = (6.62607015*10**-34)/(2*np.pi)
k_b = 1.380649*10**-23

### Predicting the Resonant Frequencies $f_0$ for 1-4mm 
The resonant frequency of a CPW resonator of $\frac{\lambda}{4}$ can be found using the followng equation[1]:
$$
f_0 = \frac{c}{\sqrt{\epsilon_{eff}}}\frac{1}{4l}
$$
where $c$ is the speed of light, $\epsilon_{eff}$ is th effective permitivity of CPW and $l$ is length of the CPW with a factor of $4$ as it is a quarter wavelength resonator which implies $2l = \lambda_0$, the fundamental mode. This uses the lumped element approximations to derive.

To calculate the the effective permeability the following equation is used[2].
$$
\epsilon_{eff} = \frac{\epsilon_r + 1}{2} + \frac{\epsilon_r - 1}{2} \frac{1}{\sqrt{1+12d/W}}
$$  
where $d$ is the height of the substrate and $W$ is the width of microstrip. For a microstrip line built on top a single substrate. We will assume this method is vaild for CPW resonator.


In [3]:
def find_effective_permeability(e_r,d,W):
    '''
    Determines the effective dielectric constant of a
    CPW when there is a single substrate
    Input:
    -   e_r = Dieletric Constant of the substrate[float] 
    -   d   = Height of substrate[m,float]
    -   W   = Width of CPW[m,float]
    Output:
    -   e_e = Effective Dieletric Constant of CPW[float]
    '''
    e_e = ((e_r+1)/(2))+((e_r-1)/(2))*(1/(np.sqrt(1+(12*d/W))))
    return e_e

def find_resonator_frequency_lumped_approx(e_r,d,W,l):
    '''
    Finds the resonant frequency of a CPW resonator defined
    on a single substrate using the lumped element approximation
    for a quarter wavelength resonator
    Input:
    -   e_r = Dielectric constant of the subtrate[float]
    -   d   = Height of substrate[m,float]
    -   W   = Width of CPW[m,float]
    -   l   = Length of CPW resonator[m,float]
    Output:
    -   f0  = Resonant Frequency of CPW resonantor[Hz,float]
    '''
    e_e = find_effective_permeability(e_r,d,W)

    f0 = (c/(np.sqrt(e_e)))*(1/(4*l))
    return f0

theo_res_f = find_resonator_frequency_lumped_approx(11.6,270*(10**-6),10*(10**-6),12*(10**-3))
theo_res_f = theo_res_f/(10**9)
print(f'{theo_res_f} GHz')

2.433917419216113 GHz


In [4]:
#Resonator Lengths
l1 = 571.248*10**-6
l2 = 648.848*10**-6
l3 = 592.769*10**-6
l4 = 680.565*10**-6
l5 = 619.493*10**-6
l6 = 550.864*10**-6
l_vals = np.array([l1,l2,l3,l4,l5,l6])

#Substrate Height
h_s = 500*10**-6

#CPW width
w_CPW = 20*10**-6

#Sapphire dielectric constant
e_sap = 8.3

In [5]:
#Find Resoant Frequencies(GHz)
f0_lumped = (find_resonator_frequency_lumped_approx(e_sap,h_s,w_CPW,l_vals))/(10**9)
print(f0_lumped)

[59.55268494 52.43038765 57.39057232 49.98692581 54.91482901 61.75635396]


#### Conformal Mapping Techniques to find $f_0$
The phase velocity $v_p = c/\sqrt{\epsilon_eff}$ which is related to the resonant frequency $f_0$. 
$$
f_0 = \frac{v_p}{4l}
$$
$$
v_p = \frac{1}{\sqrt{L_lC_l}}
$$
Using conformal mapping techniques can be used to find the geometric contribution to $L_l$ and $C_l$.
$$
L_l = \frac{\mu_0}{4} \frac{K(k'_0)}{K(k_0)} \hspace{10mm} C_l = 4\epsilon_0\epsilon_{eff} \frac{K(k_0)}{K(k'_0)}
$$

Here, $K$ denotes the complete elliptic integral of the first kind with the arguments
$$
k_0 = \frac{w}{w+2s} \hspace{10mm} k'_0 = \sqrt{1-k^2_0}
$$


In [6]:
def find_f0_Conformal_Mapping(w,s,e_r,d,W,l):
    '''
    Determine the resonant frequencies for a 
    CPW resonator using conformal mapping techniques.
    Determines the Inductance (L_l) and Capacitance (C_l)
    per unit length. Uses calculations to determine 
    effective dielectric constant of the CPW. 
    Input:
    -   w   = centre conductor width[m,float]
    -   s   = seperation of centre conductor[m,float]
    -   e_r = dieletric constant of substrate[float]
    -   d   = height of substrate[m,float]
    -   W   = width of CPW[m,float]
    -   l   = length of CPW[m,float]
    Output:
    -   L   = Magnetic Inductance of CPW[,float]
    -   C   = Capcitance of CPW[F,float]
    -   f0  = Resonant Frequency of CPW[Hz,float]
    '''
    #Find Arguements of the First Kind
    k_0 = (w)/(w+2*s)
    k_prime = np.sqrt(1-k_0**2)

    #Find Effective Permeability
    e_eff = find_effective_permeability(e_r,d,W)

    #Elleptical Integrals
    L_l = ((mu_0)/(4))*(ellipk(k_prime))/(ellipk(k_0))
    C_l = 4*epsilon*e_eff*((ellipk(k_0))/(ellipk(k_prime)))

    #Find Resonant Frequencies
    f0 = ((1/np.sqrt(L_l*C_l))/(4*l))/(10**9)

    return L_l,C_l,f0


In [7]:
#Dimensions of CPW
central_width = 10*(10**-6)
seperation = 5*(10**-6)

#Apply Conformal Mapping
L_l_cm,C_l_cm,f0_CM = find_f0_Conformal_Mapping(central_width,seperation,e_sap,h_s,w_CPW,l_vals)

print('Resonant Frequencies using Conformal Mapping (GHz)')
print(f0_CM)
print('---------')
print('Resonant Frequencies using Lumped Approximation (GHz)')
print(f0_lumped)

Resonant Frequencies using Conformal Mapping (GHz)
[59.51211719 52.39467167 57.35147742 49.95287433 54.8774206  61.71428505]
---------
Resonant Frequencies using Lumped Approximation (GHz)
[59.55268494 52.43038765 57.39057232 49.98692581 54.91482901 61.75635396]


### Discussion
These values of the Resonant Frequencies are in agreement with each other as expected as both use the same value of the $\epsilon_{eff}$ to determine them and are in agreement in their values of $v_p$ the phase velocity. These values of the resonant frequency are valid assuming there is no contributions from kinetic inductance involved. Therefore an KeysightADS EM simulation of the circuit using PEC and the conductor and with no inputted is a valid method of verifying these theortical results.

### Introducing Kinetic Inductance
When a superconducting wire is suffciently thin, the kinetic inductance $L_K$ dominates over the magnetic inductance $L_m$ as calculated in conformal mapping. This occurs when the width $w$ and thickness $t$ of CPW resonantor $<<\lambda_L$ the london peneteration depth.
For the case of a homogenous superconducting cylinder of uniform circular or rectangular cross section. The kinetic inductance can be expressed as  
$$
L_K = \mu_0\lambda_L^2(T)\frac{l}{A}
$$  
which depends on the length of the $l$ of the wire, the area $A$, and the london penertation depth $\lambda_L$ which is a function of temperature. In this circuit design we will be using Niobium Nitride (NbN) at a thickness of 10nm. NbN $\lambda_L(0) = 230 \pm 20 \mathrm{nm}$ and $T_c = 14.8 \pm 0.3 K$. This is only vaid for $T$ near $T_C$, in 1D and for $I\approx0$. Therefore, with a wire of thickness $t=10 \mathrm{nm}$ and central conductor width $w=10 \mathrm{\mu m}$. The peneteration depth varies with temperature as follows:
$$
\lambda_L(T) = \frac{\lambda_L(0)}{\sqrt{1-\left(\frac{T}{T_c} \right)^4}}

In [8]:
def find_Kinetic_Inductance_london(l,t,w,T,Tc,lambda_L_0):
    '''
    Determines the Kinetic inductance of CPW resonator
    of a material identified by varying the charateristic
    qualities of Critical Temperature (Tc) and London
    Peneteration Depth at Zero Temperature (lambda_L_0)
    Input:
    -   l           = Length of CPW resonator[m,float]
    -   t           = Thickness of conductor[m,float]
    -   w           = Width of centre conductor[m,float]
    -   T           = Temperature of System[K,float]
    -   Tc          = Critical Temperature of Material [K,float]
    -   lambda_L_0  = London Peneteration depth of Material at 0K [K,float]
    Output:
    -   L_K         = Kinetic Inductance of CPW resonator[float]
    '''
    lambda_L = (lambda_L_0)/(np.sqrt(1-(T/Tc)**4))

    L_K = mu_0*(lambda_L**2)*(l/(t*w))
    return L_K

def find_Kinetic_Inductance_Resistance(t,w,Tc,rho_n):
    normal_state_sheet_resistance = rho_n/t

    L_K_l = (0.18*hbar*normal_state_sheet_resistance)/(k_b*Tc*w)
    return L_K_l

In [9]:
thickness = 10*10**-9
Temp = 10*10**-3
NbN_Tc = 9
NbN_lambda_L = 230*10**-9
NbN_rho= 1.075*10**-5


L_K_NbN_lon = find_Kinetic_Inductance_london(l_vals,thickness,central_width,Temp,NbN_Tc,NbN_lambda_L)
L_K_l_NbN_res = find_Kinetic_Inductance_Resistance(thickness,central_width,NbN_Tc,NbN_rho)

print('Kinetic Inductance of NbN Nanowires using London Parameters')
print(L_K_NbN_lon)
print('--------')
print('Kinetic Inductance of NbN Nanowires using Resistivity')
print(L_K_l_NbN_res*l_vals)
print('--------')
print('Magentic Inductance of NbN Nanowiresn')
print(L_l_cm*l_vals)

Kinetic Inductance of NbN Nanowires using London Parameters
[3.79743395e-10 4.31328849e-10 3.94049716e-10 4.52413074e-10
 4.11814790e-10 3.66192907e-10]
--------
Kinetic Inductance of NbN Nanowires using Resistivity
[9.38114894e-09 1.06555117e-08 9.73457110e-09 1.11763746e-08
 1.01734380e-08 9.04639881e-09]
--------
Magentic Inductance of NbN Nanowiresn
[2.36306625e-10 2.68407209e-10 2.45209160e-10 2.81527495e-10
 2.56264005e-10 2.27874431e-10]


In [ ]:
L_K_l_NbN_lon = L_K_NbN_lon/l_vals

f0_kin_lon = (1/(4*l_vals*np.sqrt((L_l_cm+L_K_l_NbN_lon)*C_l_cm)))
f0_kin_res = (1/(4*l_vals*np.sqrt((L_l_cm+L_K_l_NbN_res)*C_l_cm)))

print('Resonance Frequencies with L_K using London parameters (GHz)')
print(f0_kin_lon/(10**9))
print('------')
print('Resonance Frequencies with L_K using Resistivity (GHz)')
print(f0_kin_res/(10**9))


Resonance Frequencies with L_K using London parameters (GHz)
[36.85830521 32.45017806 35.52013201 30.93787241 33.98784673 38.22219846]
------
Resonance Frequencies with L_K using Resistivity (GHz)
[9.32853093 8.21287056 8.98985041 7.83011856 8.60204173 9.67372099]


In [13]:
m_lambda_L = np.sqrt(((0.18*hbar*NbN_rho)/(mu_0*k_b*NbN_Tc))*(1-((10*-3)/NbN_Tc)**4))
print(m_lambda_L)

nan


C:\Users\Benji\AppData\Local\Temp\ipykernel_8208\1834876008.py:1: RuntimeWarning: invalid value encountered in sqrt
  m_lambda_L = np.sqrt(((0.18*hbar*NbN_rho)/(mu_0*k_b*NbN_Tc))*(1-((10*-3)/NbN_Tc)**4))
